# Import packages and configure SWMM path

In [ ]:
import os
import random
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import spotpy
from spotpy.analyser import get_simulation_fields, get_minlikeindex

from swmm_api import SwmmInput
from swmm_api.input_file.macros import delete_subcatchment
from swmm_api.output_file import VARIABLES
from swmm_api.run_swmm.run_temporary import swmm5_run_temporary

In [ ]:
# LangChain / OpenAI-Integration
from langchain_ollama import ChatOllama
from langchain.tools import tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder


In [ ]:
from langchain_core.messages import convert_to_messages

In [ ]:
from langgraph.prebuilt import create_react_agent
from langgraph_supervisor import create_supervisor

In [ ]:
import time
import resource

In [ ]:
time_start = time.perf_counter()

# Choose LLM-model

In [ ]:
llm = ChatOllama(
model="gpt-oss:20b",
temperature=0.1,
base_url="http://localhost:11434")

In [ ]:
def pretty_print_message(message, indent=False):
    pretty_message = message.pretty_repr(html=True)
    if not indent:
        print(pretty_message)
        return

    indented = "\n".join("\t" + c for c in pretty_message.split("\n"))
    print(indented)


def pretty_print_messages(update, last_message=False):
    is_subgraph = False
    if isinstance(update, tuple):
        ns, update = update
        if len(ns) == 0:
            return

        graph_id = ns[-1].split(":")[0]
        print(f"Update from subgraph {graph_id}:")
        print("\n")
        is_subgraph = True

    for node_name, node_update in update.items():
        update_label = f"Update from node {node_name}:"
        if is_subgraph:
            update_label = "\t" + update_label

        print(update_label)
        print("\n")

        if not isinstance(node_update, dict):

            continue

        msgs_raw = node_update.get("messages")
        if not msgs_raw:

            continue
        
        messages = convert_to_messages(node_update["messages"])
        if last_message:
            pass

        for m in messages:
            pretty_print_message(m, indent=is_subgraph)
        print("\n")

In [ ]:
# fixed seeds for reproduction
random.seed(501)
np.random.seed(501)

@dataclass
class CalibrationState:
    """common state"""
    inp_path: str | None = None
    node_label: str | None = None

    inp_base: SwmmInput | None = None
    inp_random: SwmmInput | None = None
    ts_base_sim: pd.Series | None = None
    ts_random_sim: pd.Series | None = None
    ts_observation: pd.Series | None = None
    df_results: pd.DataFrame | None = None
    best_index: int | None = None
    best_nse: float | None = None

STATE = CalibrationState()


# Functions

## SWMM and synthetic observations

In [ ]:
def load_base_model(inp_path: str, remove_predevelop: bool = True) -> SwmmInput:
    """
    Loads a SWMM model from "inp_path"
    and optionally removes the ‘PreDevelop’ subcatchment.
    """
    inp = SwmmInput(inp_path)
    if remove_predevelop:
        delete_subcatchment(inp, "PreDevelop")
    return inp


def run_swmm_timeseries(inp: SwmmInput, node_label: str):
    """Runs SWMM and returns the inflow time series at node 'node_label'."""
    with swmm5_run_temporary(inp) as res:
        ts = res.out.get_part(label=node_label, variable=VARIABLES.NODE.TOTAL_INFLOW)
    return ts


def randomize_subcatchments(inp: SwmmInput) -> SwmmInput:
    """
    Generates a random variant of the model:
    - imperviousness randomly between 30 and 90
    - width randomly scaled
    (Based on the official example, but slightly reworded)
    """
    inp_rand = inp.copy()
    for sc in inp_rand.SUBCATCHMENTS.values():
        sc.imperviousness = random.randint(30, 90)
        sc.width = round(sc.width * random.randint(5, 15), 0)
    return inp_rand


def random_series_like(ts: pd.Series, amplitude: float) -> pd.Series:
    """Generates a random noise series with the same index length as ts."""
    noise = amplitude * (1 - 2 * np.random.rand(ts.index.size))
    return pd.Series(index=ts.index, data=noise)


def make_artificial_observation(ts_random: pd.Series) -> pd.Series:
    """
    Generates a synthetic observation from a simulated time series
    by adding relative and absolute error structures via rolling filters.
    """
    obs = ts_random.copy()

    # relative error
    obs *= 1 + random_series_like(ts_random, 0.15).rolling(4, center=True, min_periods=0).mean()
    obs *= 1 + random_series_like(ts_random, 0.15).rolling(11, center=True, min_periods=0).mean()

    # absolute error
    mean_val = obs.mean()
    obs += (mean_val * random_series_like(ts_random, 0.15)).rolling(7, center=True, min_periods=0).mean()
    obs += (mean_val * random_series_like(ts_random, 0.15)).rolling(15, center=True, min_periods=0).mean()

    return obs


## SPOTPY

In [ ]:
class SpotpyCalibrationConfig:
    """
    SPOTPY configuration:
    - Parameters to be calibrated: width_factor, perc_imp (imperviousness)
    - Target variable: NSE between simulation and observation
    """

    # Parameter definition for SPOTPY
    width_factor = spotpy.parameter.Uniform(low=0, high=1000)
    perc_imp = spotpy.parameter.Uniform(low=0, high=100)

    def __init__(self, observation: pd.Series, end_date:str, node_label:str):
        self.observation = observation
        # end of evaluation (as in the example)
        self._end_date = pd.to_datetime(end_date)
        self.node_label = node_label

    def _build_model(self, vector) -> SwmmInput:
        width_factor, perc_imp = vector

        # Copy base model
        if STATE.inp_base is None:
            raise RuntimeError("Basismodell ist nicht geladen (STATE.inp_base ist None).")

        inp = STATE.inp_base.copy()

        # change imperviousness, width_factor currently not used (can be extended)
        for sc in inp.SUBCATCHMENTS.values():
            sc.imperviousness = round(perc_imp, 1)


        return inp

    def simulation(self, vector):
        """
        Called by SPOTPY for each parameter combination.
        """
        inp = self._build_model(vector)
        sim = run_swmm_timeseries(inp, node_label=self.node_label)
        return sim

    def evaluation(self):
        """
        Delivers the observation time series to SPOTPY.
        """
        return self.observation

    def objectivefunction(self, simulation, evaluation):
        """
        Negative NSE -> SPOTPY minimized, we want to maximize NSE. 
        """
        return -spotpy.objectivefunctions.nashsutcliffe(
            evaluation.loc[: self._end_date],
            simulation.loc[: self._end_date],
        )


# Tools for agents

## Model tools

In [ ]:

@tool("prepare_swmm_model")
def prepare_swmm_model(inp_path: str, remove_predevelop: bool = True) -> str:
    """
    Loads a SWMM model, optionally removes 'PreDevelop' and saves it in the global STATE.
    """
    STATE.inp_path = inp_path
    STATE.inp_base = load_base_model(inp_path=inp_path, remove_predevelop=remove_predevelop)
    return (
        f"Base model '{inp_path}' loaded."
        f"Removed 'PreDevelop': {remove_predevelop}."
        f"Number of Subcatchments: {len(STATE.inp_base.SUBCATCHMENTS)}"
    )


@tool("generate_synthetic_observation")
def generate_synthetic_observation(node_label: str) -> str:
    """
    Creates a random model variant, simulates the base and random models,
    and generates a synthetic observation. Results are stored in STATE.
    """
    if STATE.inp_base is None:
        raise RuntimeError("Base model is not loaded. Please use prepare_swmm_model first.")

    inp_rand = randomize_subcatchments(STATE.inp_base)
    STATE.inp_random = inp_rand

    STATE.ts_base_sim = run_swmm_timeseries(STATE.inp_base, node_label=node_label)
    STATE.ts_random_sim = run_swmm_timeseries(inp_rand, node_label=node_label)
    STATE.ts_observation = make_artificial_observation(STATE.ts_random_sim)
    STATE.node_label = node_label

    return (
        "Synthetic observations generated. "
        f"length of time series: {len(STATE.ts_observation)}. "
        f"nodes: {node_label}."
    )


## Calibration tools

In [ ]:
@tool("run_spotpy_calibration")
def run_spotpy_calibration(
    end_date: str,
    repetitions: int = 300,
    kstop: int = 100,
    like_threshold: float = 0.9,
    dbname: str = "data_interim_SCE_UA_multi_agent",
) -> str:
    """
    Performs an SCE-UA calibration with SPOTPY, saves the results
    in STATE.df_results, and calculates the best NSE.
    Parameters:
      - repetitions: Number of model runs (default 300, example: 1000 in the original)
      - kstop: termination criterion of SCE-UA
      - like_threshold: NSE threshold (as a POSITIVE value) for GLUE envelope    
    """
    if STATE.ts_observation is None:
        raise RuntimeError("No observation time series in STATE. Please call generate_synthetic_observation first.")

    cfg = SpotpyCalibrationConfig(STATE.ts_observation, end_date, STATE.node_label)

    fn = Path(f"{dbname}.csv")
    sampler = spotpy.algorithms.sceua(
        cfg,
        dbname=dbname,
        dbformat="csv",
    )

    if not fn.is_file():
        sampler.sample(
            repetitions=repetitions,
            kstop=kstop,
            pcento=1e-8,
        )

    results = spotpy.analyser.load_csv_results(dbname)
    df_results = pd.DataFrame(results)
    STATE.df_results = df_results

    # Determine the best result
    best_index, best_obj = get_minlikeindex(results, verbose=False)
    best_nse = -best_obj  # because we have minimized negative NSE

    STATE.best_index = int(best_index)
    STATE.best_nse = float(best_nse)

    # Number of "good" runs for uncertainty band
    sims_cols = get_simulation_fields(results)
    count_good = (df_results["like1"] < -like_threshold).sum()

    summary = (
        f"Calibration complete.\n"
        f"Number of simulations: {len(df_results)}\n"
        f"Best NSE: {best_nse:.3f}\n"
        f"Number of runs with NSE > {like_threshold:.2f}: {count_good}"
    )

    return summary


## Analysis tools

In [ ]:
@tool("analyze_calibration_results")
def analyze_calibration_results(like_threshold: float = 0.9) -> str:
    """
    Analyzes DF results in STATE, generates:
    - Uncertainty band (GLUE) for high NSE runs
    - Plot of observation, uncertainty band, and best-fit simulation
    Returns a textual summary.
    """
    if STATE.df_results is None or STATE.ts_observation is None:
        raise RuntimeError("No calibration results or observation data in STATE.")

    df = STATE.df_results
    obs = STATE.ts_observation

    results_rec = df.to_records(index=False)
    sim_cols = get_simulation_fields(results_rec)

    # all simulations
    sim_all = df[sim_cols].T
    sim_all.index = obs.index

    # filter for good runs
    mask_good = df["like1"] < -like_threshold
    sim_good = df.loc[mask_good, sim_cols].T
    sim_good.index = obs.index

    # uncertainty bounds
    unc_bounds = sim_good.agg(["min", "max"], axis=1)

    # best result
    if STATE.best_index is None or STATE.best_nse is None:
        best_index, best_obj = get_minlikeindex(results_rec, verbose=False)
        STATE.best_index = int(best_index)
        STATE.best_nse = float(-best_obj)

    best_ts = sim_all[STATE.best_index]

    # Plot
    fig, ax = plt.subplots(figsize=(10, 4))
    obs.plot(ax=ax, color="C1", marker=".", lw=0, ms=3, label="observation data")

    ax.fill_between(
        unc_bounds.index,
        unc_bounds["min"],
        unc_bounds["max"],
        alpha=0.3,
        label=f"parameter uncertainty (NSE > {like_threshold:.2f})",
    )

    best_ts.plot(ax=ax, label=f"best-fit simulation (NSE={STATE.best_nse:.2f})")

    ax.set_xlabel("time")
    ax.set_ylabel("outflow [l/s]")
    ax.legend(loc="upper right")
    ax.grid(True)
    plt.tight_layout()
    plt.savefig('calib_results.png')

    return (
        f"Analysis complete. Best NSE:  {STATE.best_nse:.3f}. "
        f"Number of ‘good’ runs (NSE > {like_threshold:.2f}): {mask_good.sum()}."
    )


# Define agents

In [ ]:
# Tools, auf die der Model-Agent zugreifen darf
model_tools = [prepare_swmm_model, generate_synthetic_observation]

model_prompt = (
    "You are a simulation agent for SWMM.\n\n"
    "CRITICAL:\n"
    "- You MUST use the available tools to do your work.\n"
    "- Do NOT output Python code or step-by-step recipes.\n"
    "- Instead, call the tools until the SWMM base model is prepared and a synthetic\n"
    "  observation time series at node O2 is generated.\n"
    "- When you are done, reply with a SHORT confirmation of what was done."    
    )

model_agent = create_react_agent(
    model=llm,
    tools=model_tools,
    prompt=model_prompt,
    name="model_agent",
)


In [ ]:
calib_tools = [run_spotpy_calibration]

calib_prompt =(
    "You are a calibration agent.\n\n"
    "CRITICAL:\n"
    "- You MUST use the available tools to actually run the calibration.\n"
    "- Do NOT describe code or give instructions. Just call the tool.\n"
    "- Choose reasonable values for 'repetitions' and 'kstop' (e.g. repetitions 300–500\n"
    "  for testing), pass them to the tool, and let it run.\n"
    "- When done, reply only with the text returned by the tool."
    )


calib_agent = create_react_agent(
    model=llm,
    tools=calib_tools,
    prompt=calib_prompt,
    name="calib_agent",
)

In [ ]:
analysis_tools = [analyze_calibration_results]

analysis_prompt = (
    "You are an expert for uncertainty analysis in urban drainage models.\n"
    "CRITICAL:\n"
    "- You MUST use the tool 'analyze_calibration_results' to do your work.\n"
    "- Do NOT output code or recipes. Just call the tool and then summarize the result."
        )

analysis_agent = create_react_agent(
    model=llm,
    tools=analysis_tools,
    prompt=analysis_prompt,
    name="analysis_agent",
)

In [ ]:
supervisor = create_supervisor(
    model=llm,
    agents=[model_agent, analysis_agent, calib_agent],
    prompt=("You are a supervisor agent for an urban drainage model calibration system."
"You have three specialized sub-agents available as tools:\n"
"- model_agent: handles SWMM model and observation data.\n"
"- calibration_agent: performs SPOTPY calibration.\n"
            "- analysis_agent: analyzes the results and uncertainty."
"Plan the steps in a logical order and call up the appropriate tools."
"Your goal is to calibrate an urban drainage network and describe the quality of the calibration."
        "Assign work to one agent at a time, do not call agents in parallel.\n"
        "Assign work at least once.\n"
        "After the agents return, you MUST produce ONE final assistant message to the USER "
        "that summarizes the result in plain text. "
        "This FINAL message must NOT contain any tool calls or handoffs. "
        "Begin the final message with: 'FINAL ANSWER:'."
    ),
    add_handoff_back_messages=True,
    output_mode="full_history",
).compile()    

# Main method

In [ ]:
if __name__ == "__main__":

    for chunk in supervisor.stream(
        {
            "messages": [
                {
                    "role": "user",
                    "content":     "Calibrate the SWMM-model 'Detention_Pond_Model.inp' with a synthetic observation time series at node O2 using SPOTPY." 
    "Load and prepare the model, generate synthetic observation data and run the calibration with 2007-01-01 05:00 as end date."
    "Select appropriate settings for SPOTPY, perform the calibration, and evaluate the model quality and uncertainty.",
                }
            ]
        },
    ):
        pretty_print_messages(chunk, last_message=True)


    final_message_history = chunk["supervisor"]["messages"]
    last = final_message_history[-1]
    print("\nFINAL SUPERVISOR MESSAGE:\n")
    print(last.content)
    
    '''    
    print("\nFinal supervisor message history:")
    for m in convert_to_messages(final_message_history):
        print(m.pretty_repr())
    '''

In [ ]:
time_elapsed = (time.perf_counter() - time_start)
memMb=resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/1024.0/1024.0
print ("%5.1f secs %5.1f MByte" % (time_elapsed,memMb))